In [ ]:
%matplotlib inline
from zennit.composites import LayerMapComposite
import zennit.rules as z_rules

from crp.image import plot_grid
from crp.image import vis_opaque_img
from crp.attribution import CondAttribution
from crp.visualization import FeatureVisualization
from crp.concepts import TransformerChannelConcept

from torchvision.datasets import ImageFolder
import torchvision.transforms as T

import kagglehub

from lxt.efficient import monkey_patch, monkey_patch_zennit

from setup_crp import *
from attnlrp_timm_vit_conservative import *

def download_imagenet(dataset_name):
    # Download latest version of validation set
    # 50,000 images. 50 samples for each of 1000 classes
    IMAGENET_PATH = kagglehub.dataset_download(dataset_name)

    print("Path to dataset files:", IMAGENET_PATH) # C:\Users\lvche\.cache\kagglehub\datasets\titericz\imagenet1k-val\versions\1
    return IMAGENET_PATH


device = "cuda:0" if torch.cuda.is_available() else "cpu"

vit = timm.create_model("vit_small_patch16_224.augreg_in21k_ft_in1k", pretrained=True).to(device).eval()

# modify ViT for backward pass

if vit.cls_token is not None:
        vit.cls_token.requires_grad_(True)
if vit.pos_embed is not None:
        vit.pos_embed.requires_grad_(True)

for block in vit.blocks:
        block.attn.fused_attn = False

PATCH_MAP = {
    torch.nn.GELU: partial(patch_method, non_linear_forward, keep_original=True),
    torch.nn.Dropout: partial(patch_method, dropout_forward),
    # timm uses its own LayerNorm class
    TimmLayerNorm: partial(patch_method, layer_norm_forward),
    Attention: partial(
        patch_method, timm_attention_forward, keep_original=True
    ),
    #torch.nn.Identity: partial(patch_method, identity_forward, keep_original=True),
}

monkey_patch(
    vit,
    patch_map=PATCH_MAP,
    verbose=True,
)

monkey_patch_zennit(verbose=True)

###############
tcc = TransformerChannelConcept()

attribution = CondAttribution(
    tcc,
    vit,
    no_param_grad=True
    )

data_config = timm.data.resolve_data_config({}, model=vit)
transform = timm.data.create_transform(**data_config)
preprocessing = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

imagenet_data = download_imagenet("ifigotin/imagenetmini-1000")
imagenet_data = ImageFolder(
        root=imagenet_data,
        transform=transform
    )


composite = LayerMapComposite([
        (torch.nn.Conv2d, z_rules.Gamma(0.5)),
        (torch.nn.Linear, z_rules.Gamma(10)),
    ])

conditions = [
    {
        "y": [0],
        "blocks.11.mlp": [138]
    }
]

image = Image.open(dataset.samples[0][0])
sample = transform(image).unsqueeze(0).to(device)
# zennit requires gradients
sample.requires_grad = True

layer_names = []
for name, module in vit.named_modules():
    # Filtering out empty names (the root model itself)
    if type(module).__name__ == "Linear" and name != "head":
        layer_types.append(name)

mask_map = {name: tcc.mask for name in layer_names}

attr = attribution(sample, conditions, composite, mask_map=mask_map,
                   #start_layer="features.26"
                   )

print(torch.equal(attr[0], attr.heatmap))

print(attr.heatmap.shape)

imgify(attr.heatmap, symmetric=True, grid=False)